# Prediccion de Reingreso Hospitalario a 30 Dias

**Target**: `readmitted_30d` — clasificacion binaria (0 = no reingreso, 1 = reingreso a 30 dias)

---

### Indice
1. Feature Engineering
   - 1.1 Carga y exploracion inicial
   - 1.2 Distribucion de variables
   - 1.3 Estudio de correlaciones
   - 1.4 Valoracion de reduccion de dimensionalidad (PCA)
   - 1.5 Balanceo de los datos
   - 1.6 Funcion `build_features()` — pipeline completo
2. Entrenamiento del modelo
   - 2.1 Definicion y justificacion de modelos
   - 2.2 Tratamiento del desbalanceo de clases (SMOTE vs alternativas)
   - 2.3 Busqueda de hiperparametros
   - 2.4 Metricas train vs test
   - 2.5 Stacking
   - 2.6 Cross-validation y estabilidad
   - 2.7 Seleccion del mejor modelo
   - 2.8 Reproducibilidad

## 0. Semillas y reproducibilidad
> Fijamos todas las semillas antes de cualquier operacion para garantizar resultados reproducibles.

In [1]:
# ── Configuración de logging y helpers ──
import logging
import os
import warnings
from datetime import datetime

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

log_filename = "pipeline_icu_predictive.log"
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-7s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.FileHandler(log_filename, mode='a', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("Medical.ICUModel")
warnings.filterwarnings("ignore", category=FutureWarning)

logger.info("=" * 60)
logger.info("INICIO DEL MODELO PREDICTIVO ICU")
logger.info(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
logger.info("=" * 60)

# ── Configuración de Key Vault ──
NOMBRE_BOVEDA = "kv-medicalproyect"
NOMBRE_PUENTE = "AzureKeyVault"

# ── Helper para subir logs al Data Lake ──
STORAGE_ACCOUNT = "stproyectomastergrupo3"
def _subir_log(sufijo):
    try:
        from notebookutils import mssparkutils
        fecha = datetime.now().strftime('%Y%m%d_%H%M%S')
        ruta_local = "file://" + os.path.abspath(log_filename)
        ruta_destino = f"abfss://medicalproyect-logs@{STORAGE_ACCOUNT}.dfs.core.windows.net/{sufijo}.log"
        mssparkutils.fs.cp(ruta_local, ruta_destino)
    except Exception:
        pass

# ── Helper para notificaciones Telegram ──
def _enviar_telegram(mensaje):
    try:
        from notebookutils import mssparkutils
        import requests

        token = mssparkutils.credentials.getSecret(NOMBRE_BOVEDA, "TelegramToken", NOMBRE_PUENTE)
        chat_id = mssparkutils.credentials.getSecret(NOMBRE_BOVEDA, "TelegramChatId", NOMBRE_PUENTE)
        url = f"https://api.telegram.org/bot{token}/sendMessage"
        requests.post(url, json={"chat_id": chat_id, "text": mensaje, "parse_mode": "HTML"}, timeout=10)
    except Exception as e:
        logger.warning(f"Telegram falló: {e}")
        pass


_enviar_telegram("🚀 INICIADO: Modelo Predictivo ICU")
logger.info("Helper y Telegram configurados.")


In [2]:
import os, random
RANDOM_STATE = 42
os.environ['PYTHONHASHSEED'] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)

import numpy as np
np.random.seed(RANDOM_STATE)

import warnings
warnings.filterwarnings('ignore')

## 1. Feature Engineering
### 1.0 Imports

In [3]:
import time, hashlib
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from IPython.display import display

from sklearn.model_selection   import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline          import Pipeline
from sklearn.compose           import ColumnTransformer
from sklearn.preprocessing     import StandardScaler, OneHotEncoder
from sklearn.impute            import SimpleImputer
from sklearn.decomposition     import PCA
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model      import LogisticRegression
from sklearn.ensemble          import RandomForestClassifier, HistGradientBoostingClassifier, StackingClassifier
from sklearn.neural_network    import MLPClassifier
from sklearn.calibration       import CalibratedClassifierCV, calibration_curve
from sklearn.base              import clone
from sklearn.metrics           import (
    f1_score, precision_score, recall_score, fbeta_score,
    roc_auc_score, average_precision_score,
    brier_score_loss, classification_report,
    RocCurveDisplay, ConfusionMatrixDisplay, precision_recall_curve
)
from sklearn.metrics import make_scorer
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
from xgboost import XGBClassifier

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
TARGET = 'readmitted_30d'

# ── Configuración de Azure Storage ──
STORAGE_ACCOUNT = "stproyectomastergrupo3"
CONTAINER_RAW = "medicalproyect-raw"
BASE_SILVER = f"abfss://medicalproyect-processed@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver"

logger.info(f"Storage: {STORAGE_ACCOUNT}, Container: {CONTAINER_RAW}")

PATHS = {
    'patients':    f"{BASE_SILVER}/patients",
    'outcomes':    f"{BASE_SILVER}/outcomes",
    'medications': f"{BASE_SILVER}/medications",
    'diagnoses':   f"{BASE_SILVER}/diagnoses",
    'lab':         f"{BASE_SILVER}/lab_results",
}

### 1.1 Carga y exploracion inicial de los datos crudos

In [4]:
def _load_raw(paths):
    """Carga los 5 CSV desde ADLS usando Spark y devuelve Pandas DataFrames."""
    # Lee desde ADLS con Spark y convierte a Pandas para mantener el resto del código igual
    def _read_parquet(ruta):
        return spark.read.parquet(ruta).toPandas()

    def to_bool(df, cols):
        for c in cols:
            if c not in df.columns: continue
            if df[c].dtype == bool: df[c] = df[c].astype(int)
            elif df[c].dtype == object:
                df[c] = df[c].str.strip().str.lower().map(
                    {'true':1,'false':0,'1':1,'0':0}).astype(int)
            else: df[c] = df[c].astype(int)
        return df

    df_out  = _read_parquet(paths['outcomes'])
    df_out  = to_bool(df_out, ['icu_admission','in_hospital_death','readmitted_30d'])
    df_out['admission_date'] = pd.to_datetime(df_out['admission_date'], errors='coerce')
    df_out['days_to_readmission'] = pd.to_numeric(df_out['days_to_readmission'], errors='coerce')

    df_pat  = _read_parquet(paths['patients'])
    df_pat  = to_bool(df_pat, [c for c in df_pat.columns if c.startswith('dx_')])

    df_meds_raw = _read_parquet(paths['medications'])
    df_meds = (df_meds_raw.groupby('patient_id').agg(
        n_medications=('patient_id','count'),
        avg_adherence=('adherence_pct','mean'),
        generic_ratio=('is_generic','mean')).reset_index()
        if 'adherence_pct' in df_meds_raw.columns else df_meds_raw)

    df_dx_raw = _read_parquet(paths['diagnoses'])
    if 'visit_type' in df_dx_raw.columns:
        def _agg_dx(g):
            codes = []
            for col in ['primary_icd10','secondary_icd10s']:
                if col in g.columns:
                    codes += '|'.join(g[col].fillna('').astype(str)).split('|')
            return pd.Series({'n_visits': len(g),
                'n_emergency_visits': int((g['visit_type']=='emergency').sum()),
                'n_unique_icd': len(set(c for c in codes if c and c!='nan')),
                'n_specialties': g['provider_specialty'].nunique()
                               if 'provider_specialty' in g.columns else 0})
        df_dx = df_dx_raw.groupby('patient_id').apply(_agg_dx).reset_index()
    else: df_dx = df_dx_raw

    df_lab_raw = _read_parquet(paths['lab'])
    df_lab_raw['is_abnormal'] = df_lab_raw['is_abnormal'].map(
        lambda x: 1 if str(x).strip().lower() in
        ('true','1','high','low','h','l','abnormal') else 0)
    df_lab_raw['value'] = pd.to_numeric(df_lab_raw['value'], errors='coerce')
    df_lab_raw['delta_from_normal'] = pd.to_numeric(
        df_lab_raw['delta_from_normal'], errors='coerce')

    lab_global = df_lab_raw.groupby('patient_id').agg(
        n_lab_tests  =('patient_id','count'),
        n_abnormal   =('is_abnormal','sum'),
        pct_abnormal =('is_abnormal','mean'),
        avg_delta    =('delta_from_normal','mean'),
        max_delta    =('delta_from_normal','max')
    ).reset_index()

    top_tests = df_lab_raw['test_name'].value_counts().head(12).index
    df_piv = df_lab_raw[df_lab_raw['test_name'].isin(top_tests)].copy()
    df_piv['test_key'] = df_piv['test_name'].str.lower().str.replace(' ','_')
    lab_val   = (df_piv.groupby(['patient_id','test_key'])['value']
                 .mean().unstack().add_prefix('lab_').reset_index())
    lab_delta = (df_piv.groupby(['patient_id','test_key'])['delta_from_normal']
                 .mean().unstack().add_prefix('delta_').reset_index())
    df_lab = lab_global.merge(lab_val, on='patient_id', how='outer')
    df_lab = df_lab.merge(lab_delta, on='patient_id', how='outer')

    df = (df_out.merge(df_pat,  on='patient_id', how='left')
                .merge(df_meds, on='patient_id', how='left')
                .merge(df_dx,   on='patient_id', how='left')
                .merge(df_lab,  on='patient_id', how='left'))
    return df

df_raw = _load_raw(PATHS)
_subir_log("icu/datos_cargados")
logger.info(f"Dataset cargado: {len(df_raw):,} filas x {df_raw.shape[1]} columnas")
print(f"Dataset cargado: {len(df_raw):,} pacientes x {df_raw.shape[1]} variables")
print()

# Resumen de nulos por area
nulos = {}
for grp, cols in [
    ('Demograficos',  ['age','bmi','systolic_bp','charlson_index']),
    ('Labs',          [c for c in df_raw.columns if c.startswith('lab_') or c.startswith('delta_')]),
    ('Medicamentos',  [c for c in df_raw.columns if c.startswith('n_med') or c == 'avg_adherence']),
    ('Diagnosticos',  [c for c in df_raw.columns if c.startswith('n_visit') or c.startswith('n_em')]),
]:
    cols_present = [c for c in cols if c in df_raw.columns]
    if cols_present:
        nulos[grp] = f"{df_raw[cols_present].isnull().mean().mean()*100:.1f}%"

display(pd.DataFrame([nulos], index=['% nulos']).T.rename(columns={0:'% nulos media'}))


### 1.2 Distribucion de las variables

Se muestran las distribuciones de las variables mas relevantes agrupadas por categorias:
- **Variables demograficas y clinicas** (continuas)
- **Comorbilidades** (binarias dx_*)
- **Variables de resultado** incluyendo el target


In [5]:
fig = plt.figure(figsize=(20, 18))
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

# ── Variables numericas continuas ─────────────────────────────────────────────
num_vars = [('age','Edad (años)'), ('bmi','IMC'), ('systolic_bp','TA Sistolica'),
            ('charlson_index','Indice Charlson'), ('n_medications','N Medicamentos'),
            ('pct_abnormal','% Labs Anormales'), ('n_visits','N Visitas'),
            ('n_emergency_visits','N Urgencias')]

for idx, (col, label) in enumerate(num_vars):
    if col not in df_raw.columns: continue
    ax = fig.add_subplot(gs[idx // 4, idx % 4])
    data = df_raw[col].dropna()
    ax.hist(data[df_raw[TARGET]==0], bins=25, alpha=0.6, color='#3498db',
            label='No reingreso', density=True)
    ax.hist(data[df_raw[TARGET]==1], bins=25, alpha=0.6, color='#e74c3c',
            label='Reingreso', density=True)
    ax.axvline(data.median(), color='black', linestyle='--', linewidth=1,
               label=f'Mediana={data.median():.1f}')
    ax.set_title(label, fontweight='bold', fontsize=10)
    ax.set_ylabel('Densidad', fontsize=8)
    if idx == 0: ax.legend(fontsize=7)
    ax.text(0.98, 0.95, f'skew={data.skew():.2f}',
            transform=ax.transAxes, ha='right', va='top', fontsize=7,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.suptitle('Distribucion de variables clinicas — comparacion por clase target',
             fontsize=14, fontweight='bold')
plt.savefig('distributions.png', bbox_inches='tight')
plt.show()

# ── Comorbilidades (prevalencia por clase) ────────────────────────────────────
dx_cols = [c for c in df_raw.columns if c.startswith('dx_')]
if dx_cols:
    fig2, ax2 = plt.subplots(figsize=(14, 6))
    prev_0 = df_raw[df_raw[TARGET]==0][dx_cols].mean() * 100
    prev_1 = df_raw[df_raw[TARGET]==1][dx_cols].mean() * 100
    labels  = [c.replace('dx_','').replace('_',' ') for c in dx_cols]
    x2 = np.arange(len(labels)); w2 = 0.38
    ax2.bar(x2-w2/2, prev_0, w2, label='No reingreso (0)', color='#3498db', alpha=0.85)
    ax2.bar(x2+w2/2, prev_1, w2, label='Reingreso (1)',    color='#e74c3c', alpha=0.85)
    ax2.set_xticks(x2); ax2.set_xticklabels(labels, rotation=35, ha='right', fontsize=9)
    ax2.set_ylabel('Prevalencia (%)'); ax2.legend()
    ax2.set_title('Prevalencia de comorbilidades por clase target', fontweight='bold')
    plt.tight_layout()
    plt.savefig('comorbidities.png', bbox_inches='tight')
    plt.show()

# ── Variables categoricas ─────────────────────────────────────────────────────
fig3, axes3 = plt.subplots(1, 3, figsize=(15, 5))
for ax3, (col, title) in zip(axes3, [
    ('smoking_status','Estado tabaquico'),
    ('insurance_type', 'Tipo seguro'),
    ('exercise_level', 'Nivel ejercicio')
]):
    if col not in df_raw.columns: continue
    ct = df_raw.groupby([col, TARGET]).size().unstack(fill_value=0)
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct.plot(kind='bar', ax=ax3, color=['#3498db','#e74c3c'],
                alpha=0.85, edgecolor='white')
    ax3.set_title(title, fontweight='bold')
    ax3.set_ylabel('% pacientes'); ax3.set_xlabel('')
    ax3.tick_params(axis='x', rotation=30)
    ax3.legend(['No reingreso','Reingreso'], fontsize=8)
plt.suptitle('Variables categoricas vs target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('categoricals.png', bbox_inches='tight')
plt.show()


### 1.3 Estudio de correlaciones

Se usa correlacion de **Spearman** (robusta ante distribuciones no normales y relaciones monotanas no lineales).

**Doble analisis**:
1. Correlacion de cada feature con el target (ranking de importancia)
2. Matriz de correlacion entre los top predictores (deteccion de multicolinealidad)


In [6]:
TARGETS_EXCL = ['readmitted_30d','icu_admission','length_of_stay_days',
                'total_charges_usd','days_to_readmission','in_hospital_death',
                'icu_days','cost_per_day']
EXCL_ALWAYS  = ['patient_id','admission_date','discharge_date']

num_feats = [c for c in df_raw.select_dtypes(include='number').columns
             if c not in TARGETS_EXCL + EXCL_ALWAYS]

corrs_raw = (df_raw[num_feats + [TARGET]]
             .corr(method='spearman')[TARGET]
             .drop(TARGET).dropna()
             .sort_values(key=abs, ascending=False))

# ── Grafico 1: Correlacion de cada feature con el target ─────────────────────
top30 = corrs_raw.head(30)
bar_colors = ['#e74c3c' if v>0 else '#3498db' for v in top30.values]
y_pos = range(len(top30))

fig1, ax1 = plt.subplots(figsize=(10, 11))
ax1.barh(list(y_pos), top30.values[::-1], color=bar_colors[::-1],
         alpha=0.85, edgecolor='white')
ax1.set_yticks(list(y_pos))
ax1.set_yticklabels(top30.index[::-1], fontsize=9)
ax1.axvline(0,     color='black',   linewidth=1.2)
ax1.axvline( 0.05, color='#7f8c8d', linewidth=1, linestyle='--', alpha=0.7, label='|r|=0.05')
ax1.axvline(-0.05, color='#7f8c8d', linewidth=1, linestyle='--', alpha=0.7)
ax1.axvline( 0.10, color='#2c3e50', linewidth=1.2, linestyle=':', alpha=0.8, label='|r|=0.10')
ax1.axvline(-0.10, color='#2c3e50', linewidth=1.2, linestyle=':', alpha=0.8)
for i, v in enumerate(top30.values[::-1]):
    ax1.text(v+(0.002 if v>=0 else -0.002), i, f'{v:.3f}',
             va='center', ha='left' if v>=0 else 'right', fontsize=7.5)
ax1.legend(fontsize=9)
ax1.set_xlabel('Correlacion de Spearman', fontsize=11)
ax1.set_title(f'Top 30 features — correlacion con {TARGET}', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('correlations_ranking.png', bbox_inches='tight', dpi=120)
plt.show()

# ── Grafico 2: Heatmap de multicolinealidad entre las top 15 features ─────────
top15_feats = corrs_raw.head(15).index.tolist()
corr_matrix = df_raw[top15_feats].corr(method='spearman')
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig2, ax2 = plt.subplots(figsize=(11, 9))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax2, mask=mask,
            linewidths=0.5, cbar_kws={'shrink':0.8})
ax2.set_title('Multicolinealidad entre las 15 features mas correlacionadas con el target',
              fontweight='bold', fontsize=12)
ax2.tick_params(axis='x', rotation=40)
plt.tight_layout()
plt.savefig('correlations_heatmap.png', bbox_inches='tight', dpi=120)
plt.show()

# ── Resumen ───────────────────────────────────────────────────────────────────
CORR_THRESHOLD = 0.03
TOP_CORR_FEATURES = corrs_raw[corrs_raw.abs() >= CORR_THRESHOLD].index.tolist()

display(pd.DataFrame({
    'Correlacion maxima |r|':      [f"{corrs_raw.abs().max():.4f} ({corrs_raw.abs().idxmax()})"],
    'Features con |r| > 0.05':     [(corrs_raw.abs() > 0.05).sum()],
    'Features con |r| > 0.10':     [(corrs_raw.abs() > 0.10).sum()],
    'Features utiles (|r|>=0.03)': [len(TOP_CORR_FEATURES)],
}, index=['Resultado']).T)


### 1.3b Estrategia de seleccion de features

Con correlaciones maximas |r|~0.12 y muchas features de ruido, se aplican dos etapas:

```
Todas las features (~80 numericas)
        ↓  Filtro Spearman |r| >= 0.03
Features con señal util (~32)
        ↓  SelectKBest (mutual_info_classif, k=15)
15 features finales para entrenamiento
```

Justificacion de k=15: con señal debil, añadir mas features introduce ruido que perjudica la generalizacion. La regla orientativa es k ≈ sqrt(n_muestras) / señal; con |r|_max=0.125 y ~8.800 muestras de entrenamiento, 15 features ofrecen el mejor compromiso señal/dimension.

### 1.4 Valoracion de reduccion de dimensionalidad — PCA

Se evalua si PCA es beneficioso para este dataset.

**Conclusion anticipada**: con correlaciones maximas de |r|~0.125 y muchas features
de ruido, PCA mezcla la señal util con el ruido. Se opta por **SelectKBest**
(seleccion directa de las K features mas informativas respecto al target)
en lugar de PCA como paso de preprocesamiento.
PCA se muestra aqui como analisis diagnostico.


In [7]:
# Preparar matrix numerica para PCA
feat_cols_pca = [c for c in df_raw.select_dtypes(include='number').columns
                 if c not in TARGETS_EXCL + EXCL_ALWAYS]
X_pca_raw = df_raw[feat_cols_pca].copy()
imp_pca = SimpleImputer(strategy='median')
sc_pca  = StandardScaler()
X_pca_c = sc_pca.fit_transform(imp_pca.fit_transform(X_pca_raw))

pca_full = PCA(random_state=RANDOM_STATE).fit(X_pca_c)
cum_var  = np.cumsum(pca_full.explained_variance_ratio_)
n95 = int(np.searchsorted(cum_var, 0.95)) + 1
n80 = int(np.searchsorted(cum_var, 0.80)) + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
axes[0].plot(cum_var*100, color='#3498db', linewidth=2)
axes[0].axhline(95, color='#e74c3c', linestyle='--',
                label=f'95% varianza → {n95} componentes')
axes[0].axhline(80, color='#e67e22', linestyle='--',
                label=f'80% varianza → {n80} componentes')
axes[0].fill_between(range(len(cum_var)), cum_var*100, alpha=0.1, color='#3498db')
axes[0].set_xlabel('N componentes'); axes[0].set_ylabel('% Varianza acumulada')
axes[0].set_title('Scree Plot — Varianza acumulada', fontweight='bold')
axes[0].legend(fontsize=9); axes[0].set_xlim(0, min(60,len(cum_var)))

# Varianza por componente
axes[1].bar(range(1,21), pca_full.explained_variance_ratio_[:20]*100,
            color='#9b59b6', alpha=0.85, edgecolor='white')
axes[1].set_xlabel('Componente')
axes[1].set_ylabel('% Varianza explicada')
axes[1].set_title('Varianza por componente (top 20)', fontweight='bold')

plt.suptitle('Analisis PCA — Evaluacion de reduccion de dimensionalidad',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('pca_analysis.png', bbox_inches='tight')
plt.show()

display(pd.DataFrame({
    'Features originales':           [X_pca_c.shape[1]],
    'Componentes para 95% varianza': [n95],
    'Decision':                      ['SelectKBest — PCA mezcla señal con ruido (max |r|=0.125)']
}, index=['PCA']).T)


### 1.5 Balanceo de los datos

Se analiza el desbalanceo de la clase target y se justifica la estrategia de balanceo.

**Estrategia de compensacion del desbalanceo:**

En lugar de generar muestras sintéticas, se usan mecanismos nativos de cada modelo:

| Modelo | Mecanismo | Como funciona |
|---|---|---|
| Logistic Regression | `class_weight='balanced'` | Pondera el loss de cada clase inversamente a su frecuencia |
| Random Forest | `class_weight='balanced'` | Igual que LR, aplicado en cada arbol |
| HistGradientBoosting | `class_weight='balanced'` | Igual, integrado en el gradiente |
| XGBoost | `scale_pos_weight=neg/pos` | Multiplica el gradiente de la clase positiva |
| MLP | Oversampling manual (`np.repeat`) | Duplica ejemplos positivos antes del GridSearch |

**Ventaja**: no introduce ruido sintetico, sin hiperparametros adicionales, mas rapido.
**Limitacion**: no añade nueva informacion a la clase minoritaria — solo cambia los pesos.


In [8]:
vc = df_raw[TARGET].value_counts().sort_index()
neg, pos = vc.get(0,0), vc.get(1,0)
ratio = pos / neg

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Frecuencias
axes[0].bar(['Clase 0\n(No reingreso)', 'Clase 1\n(Reingreso)'],
            [neg, pos], color=['#3498db','#e74c3c'], alpha=0.85, edgecolor='white')
for i, v in enumerate([neg, pos]):
    axes[0].text(i, v+50, f'{v:,}\n({v/len(df_raw)*100:.1f}%)',
                 ha='center', fontweight='bold')
axes[0].set_title('Distribucion del target', fontweight='bold')
axes[0].set_ylabel('N pacientes')

# Pie
axes[1].pie([neg, pos], labels=[f'No reingreso\n{neg:,}', f'Reingreso\n{pos:,}'],
            colors=['#3498db','#e74c3c'], autopct='%1.1f%%',
            wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Proporcion de clases', fontweight='bold')

# Distribucion de probabilidad predicha por clase (histograma)
axes[2].hist(df_raw[df_raw[TARGET]==0]['charlson_index'].dropna(),
             bins=10, alpha=0.6, color='#3498db', label='No reingreso', density=True)
axes[2].hist(df_raw[df_raw[TARGET]==1]['charlson_index'].dropna(),
             bins=10, alpha=0.6, color='#e74c3c', label='Reingreso', density=True)
axes[2].set_xlabel('Charlson Index (principal predictor)')
axes[2].set_title('Distribucion del mejor predictor\npor clase', fontweight='bold')
axes[2].legend()

plt.suptitle(f'Analisis de Balanceo — {TARGET}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('class_balance.png', bbox_inches='tight')
plt.show()

nivel = "SEVERO" if ratio < 0.2 else ("MODERADO" if ratio < 0.4 else "LEVE")
display(pd.DataFrame({
    'Clase 0 (No reingreso)': [f"{neg:,} ({neg/len(df_raw)*100:.1f}%)"],
    'Clase 1 (Reingreso)':    [f"{pos:,} ({pos/len(df_raw)*100:.1f}%)"],
    'Ratio desbalanceo':      [f"1:{1/ratio:.1f}"],
    'Nivel':                  [nivel],
    'Estrategia':             ["class_weight + scale_pos_weight + oversampling MLP"]
}, index=['Balanceo']).T)


### 1.6 Funcion `build_features()`

Funcion unica que encapsula **todo el pipeline** desde la lectura de los CSV
hasta la preparacion de X e y para entrenamiento o prediccion.

Sigue el patron **fit/transform**:
- `fit=True` → ajusta el pipeline y transforma (para entrenamiento)
- `fit=False` → solo transforma con el pipeline ya ajustado (para prediccion/test)


### 1.5b Justificacion clinica de cada feature engineered

A continuacion se justifica **cada variable creada** en `build_features()` con su base clinica o evidencia:

---

#### Variables temporales
| Feature | Logica clinica |
|---|---|
| `admission_month` | Los reingresos son mas frecuentes en invierno (EPOC, insuficiencia cardiaca) y verano (deshidratacion). Jencks et al., NEJM 2009. |
| `admission_weekday` | Los altas de viernes/fin de semana tienen menor acceso a seguimiento ambulatorio inmediato, aumentando el riesgo de reingreso precoz. |
| `admission_quarter` | Captura estacionalidad agrupada (Q1=invierno alto riesgo, Q3=verano) cuando el mes individual tiene demasiada varianza. |

---

#### Variables de carga de morbilidad
| Feature | Logica clinica |
|---|---|
| `n_comorbidities` | Suma de todos los dx_* activos. La carga total es uno de los predictores mas robustos de reingreso (Quan et al., Med Care 2005). |
| `high_charlson` (≥3) | Umbral clinico validado: Charlson ≥3 se asocia con mortalidad >50% a 1 año. Binarizar captura el efecto de umbral no lineal (Charlson et al., J Chronic Dis 1987). |
| `high_meds` (>P75) | Polifarmacia: por encima del percentil 75 de medicamentos aumenta el riesgo de interacciones y errores de medicacion al alta (Calcaterra et al., J Gen Intern Med 2017). |
| `high_emergency` (≥2 urgencias) | Patron de alta agudeza: ≥2 urgencias en historial indica inestabilidad clinica basal recurrente. |

---

#### Variables de funcion renal y laboratorio
| Feature | Logica clinica |
|---|---|
| `high_creatinine` (>1.2 mg/dL) | La ERC es uno de los 5 diagnosticos con mayor tasa de reingreso a 30 dias. Creatinina >1.2 mg/dL es el umbral clinico de disfuncion renal incipiente (Dharmarajan et al., JAMA 2013). |
| `high_abnormal` (>30% labs anormales) | >30% de analiticas fuera de rango indica descompensacion multiorganica activa e inestabilidad clinica al alta. |

---

#### Interacciones multiplicativas
| Feature | Logica clinica |
|---|---|
| `charlson_x_meds` | Paciente comorbido + polimedicado: riesgo compuesto mayor que la suma (sinergia entre carga cronica y carga farmacologica). |
| `charlson_x_comorbid` | Charlson pondera por gravedad; `n_comorbidities` cuenta sin ponderacion. El producto captura pacientes con muchas comorbilidades graves. |
| `charlson_x_emergency` | Combina severidad cronica con patron de urgencias (descompensaciones agudas frecuentes). |
| `charlson_x_abnormal` | Pacientes con Charlson alto + labs muy alterados: doble señal de riesgo inmediato. |
| `creatinine_x_charlson` | Marcador de sindrome cardiorrenal: ICC + disfuncion renal tiene tasas de reingreso >40% a 30 dias. |
| `abnormal_x_meds` | Labs anormales en paciente polimedicado puede indicar toxicidad farmacologica o interacciones activas. |
| `delta_x_charlson` | Desviacion maxima del rango normal × severidad cronica: instabilidad aguda sobre fondo cronico. |
| `meds_x_adherence` | Muchos medicamentos + baja adherencia: mayor riesgo de descompensacion que cada factor por separado. |

---

#### Score clinico compuesto y terminos no-lineales
| Feature | Logica clinica |
|---|---|
| `clinical_risk_score` | Score ponderado segun literatura: Charlson (40%) + creatinina alta (25%) + % labs anormales (20%) + ERC (15%). |
| `charlson_sq`, `creatinine_sq`, `risk_score_sq` | El riesgo de reingreso no crece linealmente: un Charlson=6 implica riesgo mucho mayor que el doble de Charlson=3. Los terminos cuadraticos modelan esta no-linealidad en el extremo superior. |

---
> **SelectKBest (k=40)**: de las ~60+ features totales, selecciona las 40 mas informativas respecto al target por informacion mutua, eliminando ruido y mejorando generalizacion.


In [9]:
# COLUMNAS A EXCLUIR 
DROP_ALWAYS = [
    'readmitted_30d', 'icu_admission', 'length_of_stay_days',
    'total_charges_usd', 'days_to_readmission', 'in_hospital_death',
    'icu_days', 'cost_per_day', 'patient_id', 'admission_date', 'discharge_date'
]

def build_features(paths_or_df, fit=True, pipeline=None,
                   k_features=15, random_state=RANDOM_STATE,
                   target_col='readmitted_30d', return_pipeline=True,
                   corr_threshold=0.03):
    """
    Pipeline completo: lectura de CSV → feature engineering → preprocesamiento → X, y.

    Parametros
    ----------
    paths_or_df : dict de rutas CSV  o  DataFrame ya cargado
    fit         : True  → ajustar pipeline (entrenamiento)
                  False → solo transformar (prediccion/test)
    pipeline    : sklearn Pipeline ya ajustado (requerido si fit=False)
    k_features  : numero de features a seleccionar con SelectKBest (default=15)
    corr_threshold: umbral minimo de |Spearman r| para pre-filtrar features (0=sin filtro)
    random_state: semilla para reproducibilidad
    target_col  : columna objetivo
    return_pipeline : si True, devuelve (X, y, pipeline)

    Retorna
    -------
    X          : numpy array listo para entrenar/predecir
    y          : array del target (None si no esta disponible)
    pipeline   : pipeline ajustado (si return_pipeline=True y fit=True)
    """
    # ── Paso 1: Carga y merge ─────────────────────────────────────────────────
    if isinstance(paths_or_df, dict):
        df = _load_raw(paths_or_df)
    else:
        df = paths_or_df.copy()

    # ── Paso 2: Feature Engineering ───────────────────────────────────────────
    dx_cols = [c for c in df.columns if c.startswith('dx_')]
    df['admission_month']   = df['admission_date'].dt.month   if 'admission_date' in df.columns else 0
    df['admission_weekday'] = df['admission_date'].dt.dayofweek if 'admission_date' in df.columns else 0
    df['admission_quarter'] = df['admission_date'].dt.quarter if 'admission_date' in df.columns else 0
    df['n_comorbidities']   = df[dx_cols].sum(axis=1) if dx_cols else 0

    ch = df.get('charlson_index',   pd.Series(0, index=df.index))
    cr = df['lab_creatinine'].fillna(df['lab_creatinine'].median())          if 'lab_creatinine' in df.columns else pd.Series(0, index=df.index)
    ab = df['pct_abnormal'].fillna(0) if 'pct_abnormal' in df.columns else pd.Series(0, index=df.index)
    me = df['n_medications'].fillna(0) if 'n_medications' in df.columns else pd.Series(0, index=df.index)
    em = df['n_emergency_visits'].fillna(0) if 'n_emergency_visits' in df.columns else pd.Series(0, index=df.index)
    mx = df['max_delta'].fillna(0)   if 'max_delta' in df.columns else pd.Series(0, index=df.index)

    df['high_charlson']        = (ch >= 3).astype(int)
    df['high_creatinine']      = (cr > 1.2).astype(int)
    df['high_abnormal']        = (ab > 0.3).astype(int)
    df['high_meds']            = (me > me.quantile(0.75)).astype(int) if me.std() > 0 else 0
    df['high_emergency']       = (em >= 2).astype(int)
    df['charlson_x_meds']      = ch * me
    df['charlson_x_comorbid']  = ch * df['n_comorbidities']
    df['charlson_x_emergency'] = ch * em
    df['charlson_x_abnormal']  = ch * ab
    df['creatinine_x_charlson']= cr * ch
    df['abnormal_x_meds']      = ab * me
    df['delta_x_charlson']     = mx * ch
    df['meds_x_adherence']     = me * df['avg_adherence'].fillna(0)                                   if 'avg_adherence' in df.columns else 0
    df['clinical_risk_score']  = (ch*0.40 + (cr>1.2).astype(int)*0.25
                                  + ab*0.20
                                  + df.get('dx_chronic_kidney_disease',
                                           pd.Series(0,index=df.index))*0.15)
    df['charlson_sq']          = ch ** 2
    df['creatinine_sq']        = cr ** 2
    df['risk_score_sq']        = df['clinical_risk_score'] ** 2

    # ── Paso 3: Separar X e y ─────────────────────────────────────────────────
    feat_cols = [c for c in df.columns if c not in DROP_ALWAYS]
    y = df[target_col].values if target_col in df.columns else None

    if y is not None:
        valid_mask = ~pd.isna(df[target_col])
        X_df = df.loc[valid_mask, feat_cols]
        y    = df.loc[valid_mask, target_col].values.astype(int)
    else:
        X_df = df[feat_cols]

    # ── Paso 3b: Filtro de correlacion — eliminar features con señal nula ─────────
    # Solo en fit=True: calcula Spearman |r| con target y elimina features ruido
    if fit and y is not None and corr_threshold > 0:
        from scipy.stats import spearmanr
        num_all = X_df.select_dtypes(include='number').columns.tolist()
        kept_num = []
        for col in num_all:
            vals = X_df[col].fillna(X_df[col].median())
            if vals.std() > 0:
                r, _ = spearmanr(vals, y, nan_policy='omit')
                if not __import__('numpy').isnan(r) and abs(r) >= corr_threshold:
                    kept_num.append(col)
        if not kept_num:
            kept_num = num_all  # fallback: no filtrar si todo queda vacio
        cat_all = X_df.select_dtypes(exclude='number').columns.tolist()
        X_df = X_df[kept_num + cat_all]
        print(f"  Filtro |r|>={corr_threshold}: {len(num_all)} → {len(kept_num)} features numericas")
    elif not fit and hasattr(pipeline, '_corr_kept_cols'):
        # En modo prediccion: restaurar las mismas columnas del fit
        existing = [c for c in pipeline._corr_kept_cols if c in X_df.columns]
        X_df = X_df[existing]

    # ── Paso 4: Preprocesamiento ──────────────────────────────────────────────
    if fit:
        num_cols = X_df.select_dtypes(include='number').columns.tolist()
        cat_cols = X_df.select_dtypes(exclude='number').columns.tolist()

        num_pipe = Pipeline([('imp', SimpleImputer(strategy='median')),
                             ('sc',  StandardScaler())])
        cat_pipe = Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                             ('ohe', OneHotEncoder(handle_unknown='ignore',
                                                   sparse_output=False))])
        transformers = [('num', num_pipe, num_cols)]
        if cat_cols: transformers.append(('cat', cat_pipe, cat_cols))
        prep = ColumnTransformer(transformers, remainder='drop')

        selector = SelectKBest(mutual_info_classif,
                               k=min(k_features, X_df.shape[1]))
        pipeline = Pipeline([('prep', prep), ('sel', selector)])

        if y is not None:
            X = pipeline.fit_transform(X_df, y)
        else:
            X = pipeline.fit_transform(X_df)
        # Guardar columnas post-filtro para restaurarlas en modo prediccion
        pipeline._corr_kept_cols = X_df.columns.tolist()
    else:
        if pipeline is None:
            raise ValueError("pipeline no puede ser None cuando fit=False")
        X = pipeline.transform(X_df)

    if return_pipeline and fit:
        return X, y, pipeline
    return X, y


# Verificacion
X_check, y_check, pipe_check = build_features(PATHS, fit=True, k_features=30)
X_check2, _ = build_features(PATHS, fit=False, pipeline=pipe_check, return_pipeline=False)
display(pd.DataFrame({
    'X train shape': [X_check.shape],
    'y shape':       [y_check.shape],
    'Positivos':     [f"{y_check.sum():,} ({y_check.mean()*100:.1f}%)"],
    'X pred shape':  [X_check2.shape],
}, index=['build_features()']).T)


---
## 2. Entrenamiento del modelo

### 2.0 Preparacion de datos de entrenamiento y test

In [10]:
# k=15: las 15 features mas informativas tras filtro de correlacion Spearman |r|>=0.03
X_full, y_full, PIPELINE = build_features(PATHS, fit=True, k_features=15, corr_threshold=0.03)
logger.info(f"Pipeline final ajustado. X: {X_full.shape}, y: {y_full.shape}")

X_tr, X_te, y_tr, y_te = train_test_split(
    X_full, y_full, test_size=0.2, random_state=RANDOM_STATE, stratify=y_full)

display(pd.DataFrame({
    'Train': [f"{X_tr.shape[0]:,} muestras", f"{X_tr.shape[1]} features", f"{y_tr.sum():,} ({y_tr.mean()*100:.1f}%)"],
    'Test':  [f"{X_te.shape[0]:,} muestras", f"{X_te.shape[1]} features", f"{y_te.sum():,} ({y_te.mean()*100:.1f}%)"],
}, index=['Muestras','Features','Positivos']))

# Guardamos los patient_id del test set para guardar predicciones en Gold
logger.info("Extrayendo patient_ids del test set...")
try:
    from sklearn.model_selection import train_test_split as _refit_split
    _, _, _, _, idx_tr, idx_te = _refit_split(
        X_full, y_full, np.arange(len(y_full)),
        test_size=0.2, random_state=RANDOM_STATE, stratify=y_full
    )
    df_temp = _load_raw(PATHS)
    mask = df_temp[TARGET].notna()
    df_valid = df_temp[mask].reset_index(drop=True)
    patient_ids_test = df_valid.iloc[idx_te]['patient_id'].values
    logger.info(f"{len(patient_ids_test)} patient_ids extraidos")
except Exception as e:
    logger.warning(f"No se pudieron extraer patient_ids: {e}")
    patient_ids_test = np.array([f'P{str(i).zfill(7)}' for i in range(len(y_te))])



### 2.1 Definicion y justificacion de los modelos

Se comparan **5 modelos** con justificacion explicita de cada eleccion:

---

#### Modelo 1 — Logistic Regression (baseline lineal)
**Justificacion**: Modelo lineal interpretable. En datasets pequeños con señal
debil, la regularizacion L2 evita sobreajuste. Sirve como baseline para saber
si los modelos no-lineales añaden valor real.
- Hiperparametro `C`: inverso de la regularizacion. Buscado por GridSearch.
- `class_weight='balanced'`: compensa el desbalanceo sin necesidad de SMOTE externo.

---

#### Modelo 2 — Random Forest
**Justificacion**: Ensemble no-lineal robusto. Maneja bien features mixtas
(numericas + categoricas), no asume normalidad, produce importancia de features.
- `n_estimators`: mas arboles = menos varianza; buscado en [200, 300].
- `max_depth=[4,5,6]`: arboles acotados — sin limite (`None`) los arboles memorizan el train.
- `min_samples_leaf=[10,20,30]`: cada hoja necesita al menos N muestras; controla granularidad.
- `max_features=['sqrt', 0.5]`: submuestreo de features por split; añade regularizacion estocastica.
- `class_weight='balanced'`: ajusta pesos de clase proporcionalmente.

---

#### Modelo 3 — HistGradientBoosting
**Justificacion**: Gradient boosting basado en histogramas (inspirado en LightGBM).
50x mas rapido que GradientBoosting clasico. Maneja NaN nativamente.
Especialmente eficaz cuando el numero de features supera al de muestras en
algunas regiones del espacio.
- `learning_rate`: controla el paso de cada arbol; lr pequeño + mas iteraciones = mejor generalizacion.
- `max_depth`: profundidad de cada arbol debil; arboles poco profundos evitan sobreajuste.
- `l2_regularization` ****: regularizacion L2 sobre los pesos de las hojas. Rango explorado: [0.1, 1.0, 10.0].
- `early_stopping=True` ****: detiene el boosting si la perdida de validacion no mejora
  en `n_iter_no_change=15` rondas, evitando sobreajuste por exceso de iteraciones.
- `min_samples_leaf` ampliado a [20,50,100]: hojas con mas muestras generalizan mejor.

---

#### Modelo 4 — Red Neuronal (MLPClassifier)
**Justificacion**: Captura interacciones no-lineales complejas entre features.
Con regularizacion L2 (`alpha`) y early stopping funciona bien en datasets de tamaño medio.

**Eleccion de la funcion de activacion de la ultima capa (sigmoid/logistic)**:
Para clasificacion binaria la ultima capa debe producir una probabilidad en [0,1].
La funcion **sigmoid (logistica)**: σ(x) = 1/(1+e^-x) cumple exactamente esto.
En sklearn MLPClassifier se activa automaticamente cuando `out_activation_='logistic'`
(detectado internamente para targets binarios).

**Eleccion de la funcion de error (log-loss / binary cross-entropy)**:
L = -[y·log(p) + (1-y)·log(1-p)]
Penaliza las predicciones equivocadas de forma asimetrica: una prediccion muy
segura y equivocada recibe penalizacion exponencial. Es la funcion natural para
modelos probabilisticos binarios y es equivalente a maximizar la log-verosimilitud.
En sklearn: `loss='log_loss'` (activado automaticamente para clasificacion).

**Arquitectura y regularizacion**:
La regla practica es que el numero de parametros de la red no supere `n_muestras / 10`.
Con ~9.000 muestras de entrenamiento el limite es ~900 parametros → arquitecturas tipo
`(32,)`, `(32,16)` o `(64,32)`. El desbalanceo se compensa con `alpha` alto (L2 fuerte)
en lugar de oversampling, evitando el gap artificial entre train y test.

- `hidden_layer_sizes`: buscado en [(32,), (32,16), (64,32)] — arquitecturas contenidas.
- `alpha`: buscado en [0.1, 1.0, 5.0, 10.0] — regularizacion L2 fuerte.
- `early_stopping=True` con `validation_fraction=0.15`: detiene el entrenamiento cuando
  la perdida de validacion no mejora en 30 iteraciones consecutivas.
- `activation='relu'`: rapida, evita vanishing gradient, funciona bien en capas ocultas.
---

#### Modelo 5 — XGBoost
**Justificacion**: Gradient boosting con regularizacion L1 (`reg_alpha`) y L2 (`reg_lambda`)
explicitas y soporte nativo de desbalanceo mediante `scale_pos_weight`. A diferencia de
HistGradientBoosting, XGBoost permite controlar independientemente la esparsidad (L1) y
el shrinkage (L2), lo que ofrece mayor flexibilidad para datasets con features correlacionadas.

**Desbalanceo de clases**: `scale_pos_weight` multiplica el gradiente de la clase positiva,
equivalente a `class_weight='balanced'` pero integrado en el algoritmo de boosting. Se busca
el valor optimo mediante GridSearchCV en el rango [2.0, 3.0, 4.0, 5.0].

- `scale_pos_weight`: ratio neg/pos que compensa el desbalanceo — buscado en GridSearch.
- `reg_alpha` (L1): promueve esparsidad en los pesos; util con features redundantes.
- `reg_lambda` (L2): shrinkage de los pesos de las hojas — principal control de sobreajuste.
- `subsample` y `colsample_bytree`: submuestreo estocastico de filas y columnas por arbol;
  añaden regularizacion similar al dropout de las redes neuronales.
- `max_depth=[3,4]`: arboles superficiales para evitar sobreajuste con señal debil.


### 2.2 Estrategia de balanceo de clases

El dataset tiene un desbalanceo severo: **85,1% negativos / 14,9% positivos** (ratio 1:5,7).
Sin compensacion, todos los modelos convergen a predecir siempre la clase mayoritaria
obteniendo 85% de accuracy con Recall=0%.

#### SMOTE — evaluado y descartado

Se evaluo el uso de **SMOTE** (*Synthetic Minority Over-sampling TEchnique*), que genera
pacientes sinteticos interpolando entre pacientes reales de la clase minoritaria. Se descarto
por dos razones:

- **Dependencia externa**: requiere `imbalanced-learn`, añadiendo una libreria solo para este paso.
- **Riesgo de contaminacion de folds**: debe aplicarse estrictamente dentro de cada fold de CV.
  Si se aplica antes del split, muestras sinteticas derivadas del mismo paciente aparecen en
  train y validacion simultaneamente, inflando el CV-AUC artificialmente
  (error detectado en versiones previas: CV-AUC=0.89 vs AUC test=0.56).
- **Sin mejora sobre alternativas nativas**: los experimentos comparativos mostraron que
  `class_weight='balanced'` produce resultados equivalentes o mejores sin estos riesgos.

#### Mecanismo elegido por modelo

| Modelo | Mecanismo | Justificacion |
|---|---|---|
| Logistic Regression | `class_weight='balanced'` | Pondera el loss inversamente a la frecuencia de clase |
| Random Forest | `class_weight='balanced'` | Idem, aplicado en cada arbol del ensemble |
| HistGradientBoosting | `class_weight='balanced'` | Integrado en el calculo del gradiente |
| XGBoost | `scale_pos_weight` en GridSearch | Multiplicador del gradiente; valor optimo buscado por CV |
| Red Neuronal (MLP) | alpha alto [0.1–10.0] | MLPClassifier no soporta `class_weight`; la regularizacion L2 fuerte actua como compensacion |
| Stacking | Hereda de modelos base | Meta-modelo LR usa `class_weight='balanced'` |


### 2.3 Busqueda de hiperparametros — GridSearchCV

Se usa `scoring='roc_auc'` en todos los GridSearchCV: es umbral-independiente y mas estable que F2-score o Recall para la busqueda de hiperparametros. Las metricas clinicas (Recall, F2) se evaluan en la seccion 2.4 con umbral optimizado.

`StratifiedKFold(n_splits=5)` preserva la proporcion de clases en cada fold.

In [11]:
cv5       = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
f2_scorer = make_scorer(fbeta_score, beta=2, zero_division=0)

# ── Logistic Regression ───────────────────────────────────────────────────────
t0 = time.perf_counter()
lr_search = GridSearchCV(
    Pipeline([('clf', LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE))]),
    param_grid={'clf__C': [0.01, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0, 5.0]},
    cv=cv5, scoring='roc_auc', n_jobs=-1, refit=True)
lr_search.fit(X_tr, y_tr)
t_lr = time.perf_counter() - t0
lr_best = lr_search.best_estimator_
print(f"LR         → C={lr_search.best_params_['clf__C']}  |  CV-AUC={lr_search.best_score_:.4f}  ({t_lr:.1f}s)")

# ── Random Forest ─────────────────────────────────────────────────────────────
t0 = time.perf_counter()
rf_search = GridSearchCV(
    Pipeline([('clf', RandomForestClassifier(
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))]),
    param_grid={'clf__n_estimators':     [200, 300],
                'clf__max_depth':        [4, 5, 6],       # sin None: arboles acotados
                'clf__min_samples_leaf': [10, 20, 30],    # mas conservador
                'clf__max_features':     ['sqrt', 0.5]},  # submuestreo de features
    cv=cv5, scoring='roc_auc', n_jobs=-1, refit=True)
rf_search.fit(X_tr, y_tr)
t_rf = time.perf_counter() - t0
rf_best = rf_search.best_estimator_
print(f"RF         → {rf_search.best_params_}  |  CV-AUC={rf_search.best_score_:.4f}  ({t_rf:.1f}s)")

# ── HistGradientBoosting ──────────────────────────────────────────────────────
t0 = time.perf_counter()
hgb_search = GridSearchCV(
    Pipeline([('clf', HistGradientBoostingClassifier(
        class_weight='balanced', random_state=RANDOM_STATE,
        early_stopping=True, n_iter_no_change=15))]),
    param_grid={'clf__max_iter':          [50, 100],
                'clf__max_depth':         [2, 3],
                'clf__learning_rate':     [0.01, 0.05],
                'clf__min_samples_leaf':  [20, 50, 100],
                'clf__l2_regularization': [0.1, 1.0, 10.0]},
    cv=cv5, scoring='roc_auc', n_jobs=-1, refit=True)
hgb_search.fit(X_tr, y_tr)
t_hgb = time.perf_counter() - t0
hgb_best = hgb_search.best_estimator_
print(f"HGB        → {hgb_search.best_params_}  |  CV-AUC={hgb_search.best_score_:.4f}  ({t_hgb:.1f}s)")

# ── Red Neuronal (MLP) ────────────────────────────────────────────────────────
# MLP entrena sobre X_tr original (sin oversampling) con alpha alto para regularizar.
# El oversampling 5x creaba un gap artificial: modelo entrenado en distribucion 50/50
# pero evaluado en distribucion original 15/85 → ΔAUC inflado.
t0 = time.perf_counter()
mlp_search = GridSearchCV(
    Pipeline([('clf', MLPClassifier(
        activation='relu', early_stopping=True,
        validation_fraction=0.15, n_iter_no_change=30,
        max_iter=500, random_state=RANDOM_STATE))]),
    param_grid={'clf__hidden_layer_sizes': [(32,), (32, 16), (64, 32)],
                'clf__alpha':              [0.1, 1.0, 5.0, 10.0],   # alpha alto compensa desbalanceo
                'clf__learning_rate':      ['constant', 'adaptive']},
    cv=cv5, scoring='roc_auc', n_jobs=-1, refit=True)
mlp_search.fit(X_tr, y_tr)
t_mlp = time.perf_counter() - t0
mlp_best = mlp_search.best_estimator_
mlp_clf  = mlp_best.named_steps['clf']
arch_str = ' → '.join(str(h) for h in (mlp_clf.hidden_layer_sizes if isinstance(mlp_clf.hidden_layer_sizes, tuple) else (mlp_clf.hidden_layer_sizes,)))
print(f"MLP        → arch={arch_str}  alpha={mlp_clf.alpha}  lr={mlp_clf.learning_rate}  iter={mlp_clf.n_iter_}  |  CV-AUC={mlp_search.best_score_:.4f}  ({t_mlp:.1f}s)")
print(f"             Activacion salida: {mlp_clf.out_activation_} (sigmoid — prob. binaria)  |  Loss: log_loss (binary cross-entropy)")

# ── XGBoost ───────────────────────────────────────────────────────────────────
# Regularizacion reforzada para evitar sobreajuste (ΔAUC > 0.05):
#   - base_score = prevalencia real (~0.15): evita que todas las probs esten bajo 0.5
#   - min_child_weight: exige minimo N muestras por hoja — clave con señal debil
#   - gamma: ganancia minima para realizar un split — poda arboles poco utiles
#   - max_depth reducido a [2,3]: arboles superficiales sobreajustan menos
#   - n_estimators reducido a [50,100]: menos arboles = menos memorizar el train
#   - reg_lambda alto [5,10]: shrinkage fuerte de los pesos de las hojas
t0 = time.perf_counter()
import xgboost as xgb
xgb.set_config(verbosity=0)
os.environ["PYTHONWARNINGS"] = "ignore"
xgb_search = GridSearchCV(
    Pipeline([('clf', XGBClassifier(
        base_score=float(y_tr.mean()),  # inicializa en prevalencia real (~0.15)
        eval_metric='auc',
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0))]),
    param_grid={
        'clf__scale_pos_weight': [2.0, 3.0],
        'clf__n_estimators':     [50, 100],          # reducido: menos arboles = menos overfit
        'clf__max_depth':        [2, 3],              # reducido: arboles mas superficiales
        'clf__learning_rate':    [0.01, 0.05],        # mas lento = mejor generalizacion
        'clf__min_child_weight': [5, 15, 30],         # NUEVO: minimo muestras por hoja
        'clf__gamma':            [0.1, 0.5, 1.0],     # NUEVO: ganancia minima por split
        'clf__subsample':        [0.6, 0.75],         # mas agresivo que antes
        'clf__colsample_bytree': [0.6, 0.75],
        'clf__reg_lambda':       [5.0, 10.0],         # L2 mas fuerte
    },
    cv=cv5, scoring='roc_auc', n_jobs=-1, refit=True)
xgb_search.fit(X_tr, y_tr)
t_xgb = time.perf_counter() - t0
xgb_best = xgb_search.best_estimator_
print(f"XGBoost    → {xgb_search.best_params_}  |  CV-AUC={xgb_search.best_score_:.4f}  ({t_xgb:.1f}s)")

### 2.3 Metricas en entrenamiento y test — deteccion de sobreajuste

Se reportan metricas tanto en **train** como en **test** para detectar sobreajuste.
Un gap grande (train >> test) indica que el modelo ha memorizado el training set.

**Metricas para contexto clinico:**
- **Accuracy**: incluida por completitud pero NO es la metrica principal (sesgada por desbalanceo).
- **Recall (Sensibilidad)**: fraccion de reingresos reales detectados. Objetivo: ≥70%.
- **F2-score**: F-beta con beta=2; pondera Recall 4x sobre Precision. Metrica principal.
- **Umbral clinico** (seccion 2.5): se optimiza el umbral de decision para maximizar
  el Recall manteniendo una Precision minima del 25% (por cada 4 alertas, ≥1 reingreso real).


In [12]:
all_models = {
    'Logistic Regression':  (lr_best,  t_lr,  lr_search.best_score_),
    'Random Forest':        (rf_best,  t_rf,  rf_search.best_score_),
    'HistGradientBoosting': (hgb_best, t_hgb, hgb_search.best_score_),
    'Red Neuronal (MLP)':   (mlp_best, t_mlp, mlp_search.best_score_),
    'XGBoost':              (xgb_best, t_xgb, xgb_search.best_score_),
}

# Precision minima exigida al umbral de decision.
# Significa: de cada 4 alertas lanzadas, al menos 1 debe ser un reingreso real.
# Sin este limite, modelos con probabilidades comprimidas (MLP con alpha alto)
# encuentran umbrales muy bajos (~0.20) que clasifican al 50% de pacientes
# como positivos — Recall artificialmente alto, Precision inutilmente baja.
MIN_PRECISION = 0.25

results = {}
rows    = []

for name, (model, tr_t, cv_auc) in all_models.items():
    y_tr_prob = model.predict_proba(X_tr)[:,1]
    y_te_prob = model.predict_proba(X_te)[:,1]

    # Umbral optimo con restriccion de Precision >= MIN_PRECISION:
    # entre todos los umbrales que garantizan esa precision minima,
    # se elige el que maximiza el F2-score (Recall pesa 4x sobre Precision).
    pc_tr, rc_tr, thr_tr = precision_recall_curve(y_tr, y_tr_prob)
    valid = np.where(pc_tr[:-1] >= MIN_PRECISION)[0]
    if valid.size > 0:
        f2_valid = ((1 + 2**2) * pc_tr[valid] * rc_tr[valid] /
                    (2**2 * pc_tr[valid] + rc_tr[valid] + 1e-9))
        best_thr = float(thr_tr[valid[np.argmax(f2_valid)]])
    else:
        # Fallback: ningun umbral alcanza MIN_PRECISION — usar max F1
        f1_tr = 2 * pc_tr * rc_tr / (pc_tr + rc_tr + 1e-9)
        best_thr = float(thr_tr[np.argmax(f1_tr[:-1])])

    y_tr_pred = (y_tr_prob >= best_thr).astype(int)
    y_te_pred = (y_te_prob >= best_thr).astype(int)

    auc_tr = roc_auc_score(y_tr, y_tr_prob)
    auc_te = roc_auc_score(y_te, y_te_prob)
    delta  = auc_tr - auc_te

    results[name] = dict(
        model=model, train_time=tr_t, cv_auc=cv_auc,
        threshold=best_thr,
        auc_tr=auc_tr,
        rec_tr=recall_score(y_tr, y_tr_pred, zero_division=0),
        f2_tr=fbeta_score(y_tr, y_tr_pred, beta=2, zero_division=0),
        acc_tr=(y_tr_pred==y_tr).mean(),
        auc_te=auc_te,
        rec_te=recall_score(y_te, y_te_pred, zero_division=0),
        f1_te=f1_score(y_te, y_te_pred, zero_division=0),
        f2_te=fbeta_score(y_te, y_te_pred, beta=2, zero_division=0),
        acc_te=(y_te_pred==y_te).mean(),
        y_prob=y_te_prob, y_pred=y_te_pred,
        ap=average_precision_score(y_te, y_te_prob),
        brier=brier_score_loss(y_te, y_te_prob),
        delta_auc=delta
    )
    r = results[name]
    rows.append({
        'Modelo':       name,
        'Umbral':       f"{best_thr:.3f}",
        'AUC train':    f"{r['auc_tr']:.3f}",
        'Recall train': f"{r['rec_tr']:.3f}",
        'AUC test':     f"{r['auc_te']:.3f}",
        'Recall test':  f"{r['rec_te']:.3f}",
        'F2 test':      f"{r['f2_te']:.3f}",
        'Acc test':     f"{r['acc_te']:.1%}",
        'ΔAUC':         f"{delta:+.3f}",
        'Sobreajuste':  "⚠️" if delta > 0.05 else "✓",
    })

df_metricas = pd.DataFrame(rows).set_index('Modelo')
display(df_metricas.style.map(
    lambda v: 'color: #c0392b; font-weight: bold' if v == '⚠️' else '',
    subset=['Sobreajuste']))


### 2.5 Stacking — meta-modelo sobre los 4 clasificadores base

**Justificacion**: el stacking combina las probabilidades predichas por los 4 modelos
base (LR, RF, HGB, XGBoost) como features de entrada de un meta-modelo. Cada modelo
base comete errores distintos; el meta-modelo aprende a ponderar sus predicciones
para corregir los errores individuales.

**Implementacion**:
- **Base estimators**: los 4 modelos ya entrenados con GridSearchCV.
- **Meta-modelo**: `LogisticRegression(C=0.1, class_weight='balanced')` — lineal,
  interpretable, evita sobreajuste del stacking.
- **cv=5** en StackingClassifier: genera predicciones out-of-fold para entrenar
  el meta-modelo sin data leakage.
- **passthrough=False**: el meta-modelo solo ve las probabilidades, no las features originales.

**Resultado esperado**: mejora de 0.01–0.02 de AUC sobre el mejor modelo individual.
La señal debil del dataset limita la ganancia posible.


In [13]:
t0 = time.perf_counter()
stack_clf = StackingClassifier(
    estimators=[('lr', lr_best), ('rf', rf_best), ('hgb', hgb_best), ('xgb', xgb_best)],
    final_estimator=LogisticRegression(C=0.1, class_weight='balanced',
                                       max_iter=1000, random_state=RANDOM_STATE),
    cv=5, stack_method='predict_proba', passthrough=False, n_jobs=-1)
stack_clf.fit(X_tr, y_tr)
t_stack = time.perf_counter() - t0

cv_auc_stack = np.mean([lr_search.best_score_, rf_search.best_score_,
                        hgb_search.best_score_, xgb_search.best_score_])
all_models['Stacking'] = (stack_clf, t_stack, cv_auc_stack)

y_tr_prob_s = stack_clf.predict_proba(X_tr)[:,1]
y_tr_pred_s = stack_clf.predict(X_tr)
y_te_prob_s = stack_clf.predict_proba(X_te)[:,1]
y_te_pred_s = stack_clf.predict(X_te)
delta_s = roc_auc_score(y_tr, y_tr_prob_s) - roc_auc_score(y_te, y_te_prob_s)

results['Stacking'] = dict(
    model=stack_clf, train_time=t_stack, cv_auc=cv_auc_stack,
    auc_tr=roc_auc_score(y_tr, y_tr_prob_s),
    rec_tr=recall_score(y_tr, y_tr_pred_s, zero_division=0),
    f2_tr=fbeta_score(y_tr, y_tr_pred_s, beta=2, zero_division=0),
    acc_tr=(y_tr_pred_s==y_tr).mean(),
    auc_te=roc_auc_score(y_te, y_te_prob_s),
    rec_te=recall_score(y_te, y_te_pred_s, zero_division=0),
    f1_te=f1_score(y_te, y_te_pred_s, zero_division=0),
    f2_te=fbeta_score(y_te, y_te_pred_s, beta=2, zero_division=0),
    acc_te=(y_te_pred_s==y_te).mean(),
    y_prob=y_te_prob_s, y_pred=y_te_pred_s,
    ap=average_precision_score(y_te, y_te_prob_s),
    brier=brier_score_loss(y_te, y_te_prob_s),
    delta_auc=delta_s
)
r_s = results['Stacking']
display(pd.DataFrame({
    'AUC train': [f"{r_s['auc_tr']:.3f}"],
    'AUC test':  [f"{r_s['auc_te']:.3f}"],
    'Recall':    [f"{r_s['rec_te']:.3f}"],
    'F2':        [f"{r_s['f2_te']:.3f}"],
    'ΔAUC':      [f"{delta_s:+.3f}"],
    'Tiempo':    [f"{t_stack:.1f}s"],
}, index=['Stacking']))

### 2.6 Cross-validation 5-fold — estabilidad del modelo

In [14]:
cv_summary = {}
rows_cv = []

for name, r in results.items():
    aucs = cross_val_score(r['model'], X_tr, y_tr, cv=cv5, scoring='roc_auc', n_jobs=-1)
    cv_summary[name] = aucs
    lo, hi = aucs.mean() - 1.96*aucs.std(), aucs.mean() + 1.96*aucs.std()
    rows_cv.append({'Modelo': name, 'CV-AUC media': f"{aucs.mean():.4f}",
                    'Std': f"{aucs.std():.4f}", 'Min': f"{aucs.min():.4f}",
                    'Max': f"{aucs.max():.4f}", 'IC 95%': f"[{lo:.3f}, {hi:.3f}]"})

df_cv = pd.DataFrame(rows_cv).set_index('Modelo')
display(df_cv)

fig, ax = plt.subplots(figsize=(11, 4))
mnames  = list(results.keys())
cols_cv = ['#3498db','#2ecc71','#e74c3c','#9b59b6','#f39c12','#1abc9c']
bp = ax.boxplot([cv_summary[n] for n in mnames], patch_artist=True,
                medianprops=dict(color='black', linewidth=2))
for patch, col in zip(bp['boxes'], cols_cv):
    patch.set_facecolor(col); patch.set_alpha(0.7)
ax.set_xticks(range(1, len(mnames)+1))
ax.set_xticklabels(mnames, rotation=15, ha='right', fontsize=9)
ax.set_ylabel('ROC-AUC')
ax.set_ylim(0.45, 0.82)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Azar')
ax.set_title('Cross-validation 5-fold — ROC-AUC por modelo', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### 2.7 Seleccion del mejor modelo

**Criterio de seleccion clinico** — score compuesto ponderado:

`Score = 0.35 × Recall_test + 0.25 × AUC_test + 0.20 × CV-AUC + 0.10 × (1 – |ΔAUC|) + 0.10 × (1 – Brier)`

El Recall tiene el mayor peso porque en el ambito sanitario un **Falso Negativo** (paciente de riesgo sin seguimiento) tiene un coste clinico mucho mayor que un **Falso Positivo** (seguimiento innecesario).

**Umbral clinico**: ademas del umbral por defecto (0.5), se calcula el umbral optimo que maximiza el Recall con Precision ≥ 25% — es decir, por cada 4 alertas lanzadas al equipo clinico, al menos 1 corresponde a un reingreso real.

In [15]:
cols_m = ['#3498db','#2ecc71','#e74c3c','#9b59b6','#f39c12','#1abc9c']
mnames = list(results.keys())

# Score compuesto
scores = {}
for name, r in results.items():
    cv_m = cv_summary[name].mean()
    scores[name] = (0.35*r['rec_te'] + 0.25*r['auc_te'] +
                    0.20*cv_m + 0.10*(1-abs(r['delta_auc'])) + 0.10*(1-r['brier']))
best_model = max(scores, key=scores.get)

# Tabla de seleccion
rows_sel = []
for name, sc in sorted(scores.items(), key=lambda x: -x[1]):
    r = results[name]
    rows_sel.append({'Score': f"{sc:.4f}", 'AUC test': f"{r['auc_te']:.3f}",
                     'Recall': f"{r['rec_te']:.3f}", 'F2': f"{r['f2_te']:.3f}",
                     'Acc': f"{r['acc_te']:.1%}", 'CV-AUC': f"{cv_summary[name].mean():.4f}",
                     'ΔAUC': f"{r['delta_auc']:+.3f}", 'Brier': f"{r['brier']:.4f}",
                     'Tiempo': f"{r['train_time']:.1f}s",
                     'Mejor': '★' if name == best_model else ''})
df_sel = pd.DataFrame(rows_sel, index=[n for n,_ in sorted(scores.items(), key=lambda x: -x[1])])
display(df_sel)

# Dashboard 2x3
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'Comparativa de modelos — Prediccion de reingreso hospitalario a 30 dias',
             fontsize=13, fontweight='bold')

# ROC curves
ax = axes[0,0]
for (name, r), col in zip(results.items(), cols_m):
    RocCurveDisplay.from_predictions(y_te, r['y_prob'],
        name=f"{name} ({r['auc_te']:.3f})", ax=ax, color=col)
ax.plot([0,1],[0,1],'k--',alpha=0.4)
ax.set_title('Curvas ROC', fontweight='bold')
ax.legend(fontsize=7)

# Precision-Recall
ax = axes[0,1]
for (name, r), col in zip(results.items(), cols_m):
    pc, rc, _ = precision_recall_curve(y_te, r['y_prob'])
    ax.plot(rc, pc, linewidth=2, color=col, label=f"{name} (AP={r['ap']:.3f})")
ax.axhline(y_te.mean(), color='gray', linestyle='--', alpha=0.6, label=f'Baseline ({y_te.mean():.2f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Curvas Precision-Recall', fontweight='bold')
ax.legend(fontsize=7); ax.set_xlim(0,1); ax.set_ylim(0,1.05)

# Recall + F2 (metricas clinicas)
ax = axes[0,2]
x4 = np.arange(len(mnames)); w4 = 0.35
ax.bar(x4-w4/2, [results[n]['rec_te'] for n in mnames], w4,
       label='Recall', color='#e74c3c', alpha=0.85)
ax.bar(x4+w4/2, [results[n]['f2_te']  for n in mnames], w4,
       label='F2-score', color='#e67e22', alpha=0.85)
ax.set_xticks(x4); ax.set_xticklabels(mnames, rotation=18, ha='right', fontsize=8)
ax.set_ylim(0, 1.0); ax.legend(fontsize=9)
ax.set_title('Recall y F2-score en test\n(metricas clinicas)', fontweight='bold')

# Confusion matrix del mejor modelo (umbral F1-optimo)
ax = axes[1,0]
r_b = results[best_model]
pc_b, rc_b, thr_b = precision_recall_curve(y_te, r_b['y_prob'])
f1a_b = np.where((pc_b[:-1]+rc_b[:-1])>0, 2*pc_b[:-1]*rc_b[:-1]/(pc_b[:-1]+rc_b[:-1]), 0)
thr_f1 = thr_b[np.argmax(f1a_b)]
ConfusionMatrixDisplay.from_predictions(y_te, (r_b['y_prob']>=thr_f1).astype(int),
    ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Matriz de confusion — {best_model}\n(umbral={thr_f1:.3f})', fontweight='bold')

# Calibracion
ax = axes[1,1]
ax.plot([0,1],[0,1],'k--',linewidth=1.5,label='Calibracion perfecta')
for (name, r), col in zip(results.items(), cols_m):
    fp, mp = calibration_curve(y_te, r['y_prob'], n_bins=8)
    ax.plot(mp, fp, marker='o', linewidth=2, color=col, label=f"{name} (B={r['brier']:.3f})")
ax.set_xlabel('Prob. predicha'); ax.set_ylabel('Frac. positivos reales')
ax.set_title('Calibracion de probabilidades', fontweight='bold')
ax.legend(fontsize=7)

# Estratificacion de riesgo (mejor modelo)
ax = axes[1,2]
probs_b = r_b['y_prob']
p33, p66 = np.percentile(probs_b,33), np.percentile(probs_b,66)
risk_labels = np.where(probs_b>=p66,'Alto', np.where(probs_b>=p33,'Medio','Bajo'))
order_r = ['Bajo','Medio','Alto']
rates_r = [y_te[risk_labels==l].mean()*100 for l in order_r]
cnts_r  = [(risk_labels==l).sum() for l in order_r]
bars9 = ax.bar(order_r, rates_r, color=['#2ecc71','#e67e22','#e74c3c'], alpha=0.85, edgecolor='white')
ax.axhline(y_te.mean()*100, color='gray', linestyle='--', alpha=0.7)
for b, rate, n in zip(bars9, rates_r, cnts_r):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5,
            f'{rate:.1f}%\nn={n}', ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('Tasa de reingreso (%)'); ax.set_title(f'Estratificacion de riesgo\n{best_model}', fontweight='bold')

plt.tight_layout()
plt.savefig('dashboard_final.png', bbox_inches='tight', dpi=120)
plt.show()

# Umbral clinico
print("Umbral clinico optimo (Precision >= 25%, maximizar Recall):")
rows_thr = []
for name, r in results.items():
    pc_c, rc_c, thr_c = precision_recall_curve(y_te, r['y_prob'])
    valid = np.where(pc_c[:-1] >= 0.25)[0]
    best_i = valid[np.argmax(rc_c[valid])] if len(valid) else np.argmax(rc_c[:-1])
    thr_clin = thr_c[best_i]
    y_clin = (r['y_prob'] >= thr_clin).astype(int)
    rows_thr.append({'Umbral': f"{thr_clin:.3f}",
                     'Recall': f"{recall_score(y_te, y_clin, zero_division=0):.3f}",
                     'Precision': f"{precision_score(y_te, y_clin, zero_division=0):.3f}",
                     'F2': f"{fbeta_score(y_te, y_clin, beta=2, zero_division=0):.3f}"})
df_thr = pd.DataFrame(rows_thr, index=list(results.keys()))
display(df_thr)

### 2.8 Reproducibilidad

In [16]:
import sys, sklearn, matplotlib

r_b2 = results[best_model]
display(pd.DataFrame({
    'Dataset':      [f"{X_full.shape[0]:,} pacientes"],
    'Positivos':    [f"{int(y_full.sum()):,} ({y_full.mean()*100:.1f}%)"],
    'Features':     [f"{X_full.shape[1]} (SelectKBest k=15, filtro |r|>=0.03)"],
    'Split':        ["80/20 estratificado"],
    'Mejor modelo': [best_model],
    'AUC test':     [f"{r_b2['auc_te']:.4f}"],
    'Recall test':  [f"{r_b2['rec_te']:.4f}"],
    'CV-AUC':       [f"{cv_summary[best_model].mean():.4f} ± {cv_summary[best_model].std():.4f}"],
    'Brier score':  [f"{r_b2['brier']:.4f}"],
    'Umbral F1':    [f"{thr_f1:.3f}"],
    'RANDOM_STATE': [RANDOM_STATE],
}, index=['Resumen']).T)

print()
display(pd.DataFrame({
    'Python':       [sys.version.split()[0]],
    'pandas':       [pd.__version__],
    'numpy':        [np.__version__],
    'scikit-learn': [sklearn.__version__],
    'matplotlib':   [matplotlib.__version__],
}, index=['Version']).T)

print()
print("Hashes MD5 de los CSV:")
for name_h, path_h in PATHS.items():
    try:    h = len(path_h)  # Hash no disponible: archivos en ADLS
    except: h = "no encontrado"
    print(f"  {name_h:12s}: {h}")

In [17]:
# ── Guardamos resultados en capa Gold ──
import numpy as np
import pandas as pd

GOLD_PATH = f"abfss://medicalproyect-curated@{STORAGE_ACCOUNT}.dfs.core.windows.net/gold/machine_learning"

r = results[best_model]

# 1) Predicciones del test con patient_id
df_preds = pd.DataFrame({
    'patient_id': patient_ids_test[:len(y_te)],
    'actual': y_te,
    'prediction': r['y_pred'],
    'probability': r['y_prob'],
    'model': best_model
})
df_preds['correct'] = (df_preds['prediction'] == df_preds['actual']).astype(int)

# 2) Métricas del modelo
df_metrics = pd.DataFrame([{
    'model': name,
    'auc_cv': r.get('cv_auc', 0),
    'auc_test': r.get('auc_te', 0),
    'accuracy_test': r.get('acc_te', 0),
    'recall_test': r.get('rec_te', 0),
    'f2_test': r.get('f2_te', 0),
    'threshold': r.get('threshold', 0.5),
    'training_time': r.get('train_time', 0)
} for name, r in results.items()])

logger.info(f"Guardando resultados en Gold...")
logger.info(f"  - Predicciones: {len(df_preds)} filas")
logger.info(f"  - Métricas: {len(df_metrics)} modelos")

# 3) Guardar como Parquet
try:
    spark_preds = spark.createDataFrame(df_preds)
    spark_preds.write.mode("overwrite").parquet(f"{GOLD_PATH}/icu_predictions")
    
    spark_metrics = spark.createDataFrame(df_metrics)
    spark_metrics.write.mode("overwrite").parquet(f"{GOLD_PATH}/icu_metrics")
    
    logger.info("✅ Resultados guardados correctamente en Gold")
except Exception as e:
    logger.warning(f"No se pudieron guardar resultados en Gold: {e}")

_subir_log("icu/resultados_gold")


In [18]:
spark.stop()